In [26]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from caveat.encoding.continuous import ContinuousEncoder
from caveat.mine_xz import DataModule, MutualInformationEstimator, XZDataset
from caveat.models.continuous.cvae_lstm import Encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

Device: cuda


In [27]:
def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    print(f"Found versions: {[v.name for v in versions]} in {path}")
    return Path(versions[-1])


def iter_models(path: Path):
    for dir in path.iterdir():
        if dir.is_dir() and not dir.name == "eval":
            yield latest(dir)

In [28]:
schedule_encoder = ContinuousEncoder()


def custom_loader(
    root: Path, schedule_encoder, random_z: bool = False, embed_z: bool = False
):
    for path in iter_models(root):
        xs = pd.read_csv(path / "test_inference" / "input_schedules.csv")
        xs = schedule_encoder.encode(xs, labels=None, label_weights=None)
        xs = xs.schedules

        zs = pd.read_csv(path / "test_inference" / "zs.csv", header=None).values

        if random_z:
            rng = np.random.default_rng()
            zs = rng.normal(loc=0.0, scale=1.0, size=zs.shape)

        if embed_z:
            # strong MI example
            embedder = Encoder(
                input_size=xs.shape[1] - 1,
                hidden_size=128,
                hidden_layers=2,
                dropout=0,
            )
            size = 128 * 2 * 2
            resize = nn.Linear(size, 6)
            zs = resize(embedder(xs, labels=None, hidden=None)).detach().numpy()
        yield xs, zs

Continuous Encoder initialised with:
        max_length: 12
        norm_duration: 1440
        jitter: 0
        fix_durations: stretch
        (act) weighting: unit
        (seq) joint weighting: unit
        trim eos: True
        


In [29]:
class MinerNet(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size=512,
        encoder_depth=2,
        latent_dim=6,
        block_depth=2,
        dropout=0.3,
    ):
        super(MinerNet, self).__init__()

        self.schedule_encoder = Encoder(
            input_size=input_size,
            hidden_size=hidden_size,
            hidden_layers=encoder_depth,
            dropout=dropout,
        )

        size = encoder_depth * hidden_size * 2

        self.z_embed = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=size), nn.LeakyReLU()
        )

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(size, hidden_size))
            if dropout > 0:
                blocks.append(nn.Dropout(dropout))
            blocks.append(nn.LeakyReLU())
            size = hidden_size
        self.blocks = nn.Sequential(*blocks, nn.Linear(size, 1))

    def forward(self, xs, zs):
        h1 = self.schedule_encoder(xs, labels=None, hidden=None)
        h2 = self.z_embed(zs)
        return self.blocks(h1 + h2)

In [30]:
data_loaders = {
    "random": custom_loader(
        Path("../logs/actvae/cpvae"), schedule_encoder, random_z=True
    ),
    "cpvae": custom_loader(Path("../logs/actvae/cpvae"), schedule_encoder),
    "cvae": custom_loader(Path("../logs/actvae/cvae"), schedule_encoder),
    "vae": custom_loader(Path("../logs/actvae/vae"), schedule_encoder),
    # "strong": custom_loader(
    #     Path("../logs/actvae/vae"), schedule_encoder, embed_z=True
    # ),
}
results = {}
for name, loader in data_loaders.items():
    print(f"Evaluating {name}...")
    model_results = []
    for i, (xs, zs) in enumerate(loader):

        logger = TensorBoardLogger("logs/xz", name=f"{name}_{i}")
        dataset = XZDataset(xs=xs, zs=zs)
        loader = DataModule(
            dataset=dataset,
            val_split=0.1,
            test_split=0.1,
            batch_size=1024,
            num_workers=8,
            pin_memory=False,
        )

        net = MinerNet(
            input_size=xs.shape[1] - 1,
            hidden_size=512,
            encoder_depth=3,
            block_depth=3,
            latent_dim=6,
            dropout=0.2,
        )

        kwargs = {"alpha": 1, "lr": 1e-3, "weight_decay": 1e-3}
        model = MutualInformationEstimator(net=net, **kwargs)
        trainer = Trainer(
            min_epochs=10,
            max_epochs=500,
            accelerator=device,
            devices=1,
            enable_progress_bar=False,
            logger=logger,
            enable_checkpointing=True,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20),
                ModelCheckpoint(
                    monitor="val_loss", save_top_k=2, save_weights_only=False
                ),
            ],
        )
        trainer.fit(model, datamodule=loader)
        mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
        model_results.append(mi)
    results[name] = {
        "mean": np.mean(model_results),
        "var": np.var(model_results),
    }

Evaluating random...
Found versions: ['version_0'] in ../logs/actvae/cpvae/cpvae-add_nrun0


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  8.2 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  8.2 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 8.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.2 M                                                                                                
Total estimated model params size (MB): 32                                                                         
Modules in train mode: 18                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │  -0.0040392279624938965   │
│          test_mi          │   0.0040392279624938965   │
└───────────────────────────┴───────────────────────────┘

...

Found versions: ['version_0'] in ../logs/actvae/vae/vae_nrun2


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  8.2 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  8.2 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 8.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.2 M                                                                                                
Total estimated model params size (MB): 32                                                                         
Modules in train mode: 18                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -2.418227195739746     │
│          test_mi          │     2.418227195739746     │
└───────────────────────────┴───────────────────────────┘

In [31]:
for name, result in results.items():
    print(f"\tResults for {name}: {result}")

# Results for random: {'mean': 0.00819052520673722, 'var': 1.6856190372035206e-05}
# 	Results for cvae: {'mean': 1.9896929502487182, 'var': 0.04077908030939398}
# 	Results for vae: {'mean': 2.3704758644104005, 'var': 0.03160935809538841}
# 	Results for strong: {'mean': 2.290478467941284, 'var': 0.06955916272613649}

	Results for random: {'mean': np.float64(0.006559552252292633), 'var': np.float64(2.883246047998611e-05)}
	Results for cpvae: {'mean': np.float64(2.3222882032394407), 'var': np.float64(0.13207414560223243)}
	Results for cvae: {'mean': np.float64(1.625468873977661), 'var': np.float64(0.29822956449993054)}
	Results for vae: {'mean': np.float64(2.58513503074646), 'var': np.float64(0.044931646568384165)}


In [32]:
df = pd.DataFrame.from_dict(results, orient="index")
print(df.to_latex(float_format="{:.4f}".format))

for name, result in results.items():
    print(f"\tResults for {name}: {result}")

# \begin{tabular}{lrr}
# \toprule
#  & mean & var \\
# \midrule
# random & 0.0082 & 0.0000 \\
# cvae & 1.9897 & 0.0408 \\
# vae & 2.3705 & 0.0316 \\
# strong & 2.2905 & 0.0696 \\
# \bottomrule
# \end{tabular}

\begin{tabular}{lrr}
\toprule
 & mean & var \\
\midrule
random & 0.0066 & 0.0000 \\
cpvae & 2.3223 & 0.1321 \\
cvae & 1.6255 & 0.2982 \\
vae & 2.5851 & 0.0449 \\
\bottomrule
\end{tabular}

	Results for random: {'mean': np.float64(0.006559552252292633), 'var': np.float64(2.883246047998611e-05)}
	Results for cpvae: {'mean': np.float64(2.3222882032394407), 'var': np.float64(0.13207414560223243)}
	Results for cvae: {'mean': np.float64(1.625468873977661), 'var': np.float64(0.29822956449993054)}
	Results for vae: {'mean': np.float64(2.58513503074646), 'var': np.float64(0.044931646568384165)}


In [36]:
(2.585 - 2.322) / 2.585

0.10174081237911022